In [3]:
import sys
sys.path.append("..")

import time
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, precision_score

from src.preprocess import clean_text, prepare_data, split_data

df = pd.read_csv("../data/raw.csv")
df_clean = prepare_data(df)
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df_clean)

realistic = pd.read_csv("../tests/data/realistic_test.csv")
realistic["clean"] = realistic["text"].apply(clean_text)

In [1]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\projects\smart-ticket-routing\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\eisas\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
X_train_emb = encoder.encode(X_train.tolist(), batch_size=64, show_progress_bar=True)
X_val_emb = encoder.encode(X_val.tolist(), batch_size=64)
realistic_emb = encoder.encode(realistic["clean"].tolist())

print(X_train_emb.shape)

Batches:   0%|          | 0/261 [00:00<?, ?it/s]

(16678, 384)


In [5]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train_emb, y_train)

pred = model.predict(X_val_emb)
print(classification_report(y_val, pred))
print("Precision_macro:", precision_score(y_val, pred, average="macro"))

              precision    recall  f1-score   support

     ACCOUNT       1.00      1.00      1.00       768
      CANCEL       1.00      1.00      1.00       141
     CONTACT       1.00      1.00      1.00       295
    DELIVERY       0.99      0.97      0.98       245
    FEEDBACK       1.00      1.00      1.00       298
     INVOICE       1.00      1.00      1.00       244
       ORDER       0.98      1.00      0.99       458
     PAYMENT       1.00      0.99      0.99       298
      REFUND       1.00      0.99      1.00       386
    SHIPPING       1.00      1.00      1.00       292
SUBSCRIPTION       1.00      1.00      1.00       149

    accuracy                           1.00      3574
   macro avg       1.00      1.00      1.00      3574
weighted avg       1.00      1.00      1.00      3574

Precision_macro: 0.9972079580287243


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer


message = ["where is my money? i sent the shoes back 2 weeks ago"]

start = time.perf_counter()
for _ in range(100):
    emb = encoder.encode(message)
    pred = model.predict(emb)
elapsed = (time.perf_counter() - start) / 100

baseline = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ("classifier", LogisticRegression(max_iter=1000)),
])

baseline.fit(X_train, y_train)

baseline_start = time.perf_counter()
for _ in range(100):
    pred = baseline.predict(message)
baseline_elapsed = (time.perf_counter() - baseline_start) / 100
print(f"for encoder: {elapsed * 1000:.1f} ms per message")
print(f"for baseline: {baseline_elapsed * 1000:.1f} ms per message")

for encoder: 19.2 ms per message
for baseline: 1.0 ms per message


In [9]:
import sys
sys.path.append("..")

realistic_dev = pd.read_csv("../tests/data/realistic_dev.csv")

realistic_dev["clean"] = realistic_dev["text"].apply(clean_text)
realistic_dev_emb = encoder.encode(realistic_dev["clean"].tolist())

realistic_pred = model.predict(realistic_dev_emb)

precision = precision_score(realistic_dev["category"], realistic_pred, average="macro")
print("Precision_macro on realistic dev set:", precision)


Precision_macro on realistic dev set: 0.815151515151515


In [11]:
print(classification_report(realistic_dev["category"], realistic_pred, zero_division=0))

              precision    recall  f1-score   support

     ACCOUNT       0.83      1.00      0.91         5
      CANCEL       0.75      0.75      0.75         4
     CONTACT       0.80      1.00      0.89         4
    DELIVERY       0.67      1.00      0.80         4
    FEEDBACK       0.67      0.33      0.44         6
     INVOICE       1.00      0.60      0.75         5
       ORDER       0.67      0.80      0.73         5
     PAYMENT       0.75      0.60      0.67         5
      REFUND       0.83      1.00      0.91         5
    SHIPPING       1.00      1.00      1.00         5
SUBSCRIPTION       1.00      1.00      1.00         3

    accuracy                           0.80        51
   macro avg       0.82      0.83      0.80        51
weighted avg       0.81      0.80      0.79        51



In [13]:
realistic_dev["type"] = realistic_dev["type"].str.strip()
realistic_dev["pred"] = realistic_pred
realistic_dev["correct"] = realistic_dev["category"] == realistic_dev["pred"]
realistic_dev.groupby("type")["correct"].agg(["count", "mean"])

,count,mean
type,,
ambiguous,6,1.000000
angry,5,0.400000
colloquial,8,1.000000
long,8,0.875000
multi_intent,4,0.500000
no_keyword,15,0.733333
short,3,1.000000
typo,2,1.000000


In [14]:
proba = model.predict_proba(realistic_dev_emb)

realistic_dev["pred"] = model.classes_[proba.argmax(axis=1)]
realistic_dev["confidence"] = proba.max(axis=1)
realistic_dev["correct"] = realistic_dev["pred"] == realistic_dev["category"]

realistic_dev.groupby("correct")["confidence"].describe()

,count,mean,std,min,25%,50%,75%,max
correct,,,,,,,,
False,10.0,0.481919,0.185159,0.300354,0.351129,0.458250,0.535017,0.917066
True,41.0,0.785076,0.201544,0.264846,0.679081,0.857838,0.931036,0.994073


In [15]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
rows = []

for t in thresholds:
    auto = realistic_dev["confidence"] >= t
    coverage = auto.mean()
    precision_auto = realistic_dev.loc[auto, "correct"].mean()
    rows.append({"threshold": t, "coverage": coverage, "precision_auto": precision_auto})

pd.DataFrame(rows)

,threshold,coverage,precision_auto
0,0.3,0.960784,0.795918
1,0.4,0.843137,0.883721
2,0.5,0.803922,0.878049
3,0.6,0.666667,0.970588
4,0.7,0.588235,0.966667
5,0.8,0.490196,0.960000
6,0.9,0.313725,0.937500


## Model Comparison

| Model                  | Macro precision (validation) | Macro precision (realistic dev) | Accuracy (realistic dev) | ms / message (CPU) |
|------------------------|------------------------------|---------------------------------|--------------------------|--------------------|
| TF-IDF + LR (baseline) | 99.7%                        | 67.5%                           | 57%                      | 1.0                |
| MiniLM embeddings + LR | 99.7%                        | **81.5%**                       | **80%**                  | 19.2               |

**Insight: the synthetic validation set is saturated.** Both models reach 99.7% on it,
so it cannot tell us which model is better. The difference only appears on realistic
messages. From now on, model decisions are based on the **realistic dev set**.

**Insight: embeddings understand meaning, not just keywords.**
- `CANCEL` (cancellation fees): precision went from **0% to 75%**. The messages say
  "pay something extra" or "penalty", never "fee". The baseline missed all of them.
- `ACCOUNT` precision went from **31% to 83%**. The baseline used ACCOUNT (the largest
  category) as a "default" when unsure; the embedding model does not.

**Insight: speed is not a blocker.** The embedding model is about 19× slower, but at
19 ms per message, 5,000 messages per day take less than 2 minutes of CPU time.

## Error Analysis (embedding model)

**Weak spot: complaints.** `FEEDBACK` recall is only 33%, and angry messages are only
40% correct. An angry customer writes "my order arrived damaged again", and the model
routes it to ORDER or DELIVERY because of the topic words.

**Open question for the business:** should a complaint about a damaged delivery go to
the FEEDBACK team or the DELIVERY team? Some of these "errors" may be unclear labeling
rules, not model mistakes.

**Messages without keywords** (`no_keyword`): 73% correct. Better than the baseline,
but still the hardest common type.

## Confidence and Routing

| Confidence (median) | Baseline | Embeddings |
|---------------------|----------|------------|
| Wrong predictions   | 0.34     | 0.46       |
| Correct predictions | 0.61     | 0.86       |

**Insight:** the embedding model separates correct and wrong predictions much better,
so a confidence threshold is useful. But one wrong prediction has **0.92 confidence**:
no reasonable threshold can catch it.

### Coverage vs precision (realistic dev set)

| Threshold | Coverage | Precision on auto-routed |
|-----------|----------|--------------------------|
| 0.3       | 96%      | 79.6%                    |
| 0.4       | 84%      | 88.4%                    |
| 0.5       | 80%      | 87.8%                    |
| **0.6**   | **67%**  | **97.1%**                |
| 0.7       | 59%      | 96.7%                    |
| 0.8       | 49%      | 96.0%                    |
| 0.9       | 31%      | 93.8%                    |

**Decision (provisional): threshold = 0.6.** It is the lowest threshold that reaches the
97% precision target. The model can route **two out of three messages** automatically.
This is below the 80% coverage goal, but it is an honest number on realistic messages.

**Caveats:**
- The dev set is **small** (51 messages). At threshold 0.6, precision is 33 correct out
  of 34: one more mistake would bring it to 94%. We cannot reliably distinguish 0.6
  from 0.7 with this data.
- Precision **drops** at thresholds 0.7 to 0.9 instead of rising. This is caused by the
  single high-confidence mistake: as the auto-routed group gets smaller, that one error
  weighs more.
- These messages were used to make decisions, so they are a **dev set**, not a final
  test set.

**Next steps:**
- Build a larger realistic set (150 to 200 messages), split into calibration and test.
- Confirm the threshold on the larger calibration set.
- Explore ways to increase coverage (see ideas below).